# 04 - Full training on Colab (A100 / L4, fallback for Kaggle)

Runs from the browser or from VS Code with the Colab extension (kernel -> Colab -> GPU). Code, configs and
`secrets/kaggle.json` live on Google Drive `MyDrive/DataSlayer26/` (synced from the laptop by Drive desktop); the
runtime only sees Drive, not the laptop disk. If `drive.mount` fails in VS Code, run the command palette entry
`Colab: Mount Google Drive to Server...` and re-run the first cell.

Data is downloaded with the Kaggle API to the runtime's local disk (fast I/O); checkpoints, OOF and test probs are
written to Drive (`configs/colab_b.yaml`, ConvNeXt-B clip model), so they sync back to the laptop. bf16, single
process. Every test clip is predicted on its own (competition FAQ 27). ConvNeXt-B: ~25 min per fold on A100, ~8 min
per epoch on L4.

Plan: run folds 0-2 first (baseline ConvNeXt-T on the same folds: clip F1 0.615, AUC 0.649), then folds 3-4 into the
same run directory.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/DataSlayer26"
for need in ["src/training/engine.py", "configs/colab_b.yaml", "secrets/kaggle.json"]:
    if not os.path.exists(f"{ROOT}/{need}"):
        raise FileNotFoundError(f"{ROOT}/{need} missing: wait for Drive desktop sync on the laptop")
%cd {ROOT}
!pip -q install timm accelerate kaggle
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
os.environ["KAGGLE_CONFIG_DIR"] = f"{ROOT}/secrets"
os.environ["PYTHONPATH"] = f"{ROOT}/src"
if not os.path.exists("/content/data/sample_submission.csv"):
    # A cancelled download leaves a partial zip that the Kaggle CLI would skip as "up to date".
    !rm -f /content/data-slayer-4.zip
    !rm -rf /content/data
    !kaggle competitions download -c data-slayer-4 -p /content
    !unzip -q -o /content/data-slayer-4.zip -d /content/data
!ls /content/data

In [ ]:
import torch

CONFIG = "configs/colab_b.yaml"  # ConvNeXt-B clip model; configs/colab_a100.yaml = ConvNeXt-T clip baseline
SET = "train.folds=[0,1,2]"  # second session: "train.folds=[3,4]" (same run dir, CV pools all folds)
# 16 clips/step = 128 images ~ 28 GB on ConvNeXt-B; below 40 GB (L4 24 GB) keep the same effective step as 8 x 2.
if torch.cuda.get_device_properties(0).total_memory < 35 * 2**30:
    SET += " train.batch_size=8 train.grad_accum=2"
print(torch.cuda.get_device_name(0), "| SET:", SET)
if not os.path.exists("/content/cache/videos.csv"):
    !python -m cli.prepare --config {CONFIG} --set {SET} --workers 8

In [ ]:
!python -m cli.train --config {CONFIG} --set {SET}

In [ ]:
!python -m cli.predict --config {CONFIG} --set {SET}
!ls -la {ROOT}/outputs/convnext_b_t8 {ROOT}/outputs/convnext_b_t8/fold*